# Caveat / ActVAE (Shone & Hillel 2025) — 250-Epoch Benchmark Retraining Pipeline
**Target Exhibit:** Table 8 (Caveat Multi-Domain Comparison) & Figure 9 (Diurnal Overlay)  
**Objective:** Retrain ActVAE baselines to **exact 250-epoch parity** matching TUS-GAN v4, using Shone & Hillel's author-recommended `patience: 50` and `min_epochs: 50`.
**Hardware:** Kaggle Dual Tesla T4 GPUs (or Single T4/P100/V100)  
**Dataset:** ITUS 2019 Standardized Cohort ($N = 50{,}000$ schedules, $493{,}834$ train episodes)

---

### Step 1: Verify Hardware & GPU Environment

In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print(f"Device Memory  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
assert torch.cuda.is_available(), "CUDA GPU not detected! Please enable GPU in notebook settings."

### Step 2: Install Caveat & Evaluation Dependencies

In [ ]:
!pip install -q acteval pytorch-lightning pot

# Clone and install Caveat in editable mode
import os
if not os.path.exists("caveat"):
    !git clone https://github.com/big-ucl/caveat.git
!pip install -q -e caveat --no-deps
print("✓ Caveat package successfully installed!")

### Step 3: Verify Data Directory & 50,000-Schedule Cohort
Confirm that the standardized ITUS train/test schedules and Cell A/B/D synthetic schedules exist.

In [ ]:
import os
assert os.path.exists("data/real_train_schedules.csv"), "data/real_train_schedules.csv missing!"
assert os.path.exists("data/real_test_schedules.csv"), "data/real_test_schedules.csv missing!"
assert os.path.exists("data/synth_cell_a_schedules.csv"), "data/synth_cell_a_schedules.csv missing!"
print("✓ ITUS 50,000-schedule benchmark cohort verified!")
!ls -lh data/

### Step 4: Inspect Updated 250-Epoch Configuration Files

In [ ]:
!cat configs/itus_cvae_unconstrained.yaml | grep -A 4 "trainer_params:"
!cat configs/itus_cvae_soft.yaml | grep -A 4 "trainer_params:"

### Step 5: Train Unconstrained ActVAE Baseline (250 Epochs)
Trains Caveat's `CondDiscCNN2D` on 40,000 schedules for up to 250 epochs with `patience: 50`.

In [ ]:
!python train_actvae.py --config configs/itus_cvae_unconstrained.yaml

### Step 6: Train Soft-Constrained ActVAE Baseline (250 Epochs, $\lambda = 10.0$)
Trains `CondDiscCNN2DSoft` incorporating differentiable child labor penalty on decoder probabilities.

In [ ]:
!python train_actvae.py --config configs/itus_cvae_soft.yaml

### Step 7: Run Official ActEval Multi-Domain Metric Suite (Table 8)
Evaluates all 5 models (Cell A, Cell B, Cell D, ActVAE Unconstrained, ActVAE Soft) under Caveat's official evaluator.

In [ ]:
!python run_caveat_evaluation.py --data_dir data --output_dir outputs

import pandas as pd
df_table8 = pd.read_csv("outputs/table8_caveat_evaluation.csv")
print("\n=== UPDATED TABLE 8 RESULTS ===")
display(df_table8)

### Step 8: Render Publication Figure 9 (Elsevier TRC Formatted)

In [ ]:
!python make_figure9_TRC.py --data_dir data --output_dir outputs

from IPython.display import Image, display
display(Image("outputs/figure9_diurnal_participation.png", width=900))

### Step 9: Package Outputs for Download

In [ ]:
# Package outputs to root working dir for Kaggle Output download
!zip -r caveat_retrained_250epochs_outputs.zip outputs/ -x "outputs/actvae_logs/**/checkpoints/**"
!cp caveat_retrained_250epochs_outputs.zip /kaggle/working/ 2>/dev/null || true
!cp outputs/table8_caveat_evaluation.csv /kaggle/working/ 2>/dev/null || true
!cp outputs/figure9_diurnal_participation.pdf /kaggle/working/ 2>/dev/null || true
!cp outputs/figure9_diurnal_participation.png /kaggle/working/ 2>/dev/null || true
print("✓ Outputs zipped and copied to /kaggle/working/!")

# Generate an instant cloud backup link
print("
--- Cloud Direct Download Backup Link ---")
!curl -s -F "file=@caveat_retrained_250epochs_outputs.zip" https://0x0.st || true
